# Roman-like injection–recovery

Recover a linearly limb-darkened event **with annual parallax and correlated noise**,
including the limb-darkening coefficient `a1`.
The [reference Roman experiment](https://github.com/ShotaMiyazaki94/test_microjax/tree/979fa9b33b69c4ba89eb7ad70ebae3d23da349cc/Roman_recovery)
supplies the 72-day season, 12-minute cadence (8640 observations), W149 source
magnitude 24.744, source fraction 0.395, and magnitude-error table.
We assume an **active low-mass dwarf** with 1% intrinsic flux RMS,
5-day rotation, and 20-day coherence. Near-infrared spot modulation in active
M dwarfs motivates this scale ([Goulding et al. 2012](https://doi.org/10.1111/j.1365-2966.2012.21932.x)).
The periodic M-dwarf variables in that study span 0.16–90 days and
9–115 mmag peak-to-peak in J; a 1% RMS sinusoid has about 31 mmag
peak-to-peak amplitude. The activity class and coherence are assumptions; the apparent magnitude alone
does not determine them. The J-band observations support the scale, rather
than a calibrated prediction for W149.

The injected event has `t0=.2`, `u0=.005`, `t_e=16.235`, `rho=.01`,
`pi_e_n=.2`, `pi_e_e=-.1`, and `a1=.5`. Times are HJD − 2458000; the fixed reference
epoch is 0, RA 266°, Dec −29°. We fit the `u0>0` branch.
This idealized single-season example uses the reference's W149 convention;
it is not a Roman performance forecast. Injection and recovery use 4096 FFT bins.


In [ ]:
%matplotlib inline
from pathlib import Path
import sys
root = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
            if (p / "pyproject.toml").exists())
sys.path.insert(0, str(root))
import matplotlib.pyplot as plt
from example._common import style
style()


In [ ]:
import jax
jax.config.update("jax_enable_x64", True)
from example.inference.code.inference import load_injection, fit_event


## Joint event and GP fit

The model is $F(t)=A(t)[f_s+g(t)]+f_b+\epsilon(t)$:
**the source variability is magnified too**. A damped-oscillator GP provides
a quasi-periodic approximation to disk-integrated spot modulation, with
rotation period $P$, coherence time $\tau$, and intrinsic flux RMS $\sigma$:

$$k(\Delta t)=\sigma^2 e^{-|\Delta t|/\tau}
\left[\cos\left(\frac{2\pi|\Delta t|}{P}\right)
+\frac{P}{2\pi\tau}\sin\left(\frac{2\pi|\Delta t|}{P}\right)\right].$$

The observed-flux covariance is $A(t_i)A(t_j)k(t_i-t_j)$ plus white noise.
For the chosen source fraction, 1% source RMS corresponds to about 4.3 mmag
at baseline, approaching 10.9 mmag at high magnification. Individual spot
crossings require a spatial brightness model and are outside this example.

[tinygp](https://tinygp.readthedocs.io/en/stable/) marginalizes the latent GP
in the likelihood.

All twelve event, parallax, limb-darkening, flux, and GP parameters are inferred jointly.
Independent Gaussian priors are specified below; logarithms are natural.

| Coordinate | Mean | Standard deviation |
| --- | --- | --- |
| `t0` | 0 | .5 day |
| `log(u0)` | `log(.01)` | 1 |
| `log(t_e)` | `log(15)` | .6 |
| `log(rho)` | `log(.01)` | .7 |
| `log(f_s)` | 0 | 1 |
| `f_b` | 0 | 5 |
| `log(gp_sigma)` | `log(.01)` | 1 |
| `log(gp_scale)` | `log(20)` | .7 |
| `pi_e_n`, `pi_e_e` | 0 | .5 each |
| `logit(a1)` | 0 | 1 |
| `log(gp_period)` | `log(5)` | .7 |

Here `logit(a1) = log(a1 / (1 - a1))`; the resulting logit-normal prior
keeps `0 < a1 < 1`. The intensity profile is $I(\mu) \propto 1-a_1(1-\mu)$.

`gp_sigma` is in unmagnified source-flux units. `gp_scale` and `gp_period`
are the coherence time and rotation period in days.


In [ ]:
model, data = load_injection()
result, point = fit_event(model, data.times, data.flux, data.error)
assert result.success, result.message


## Posterior and residuals

Generate the data, joint fit, and two NUTS chains from the repository root:

```bash
python example/inference/code/inference.py --hmc
```

The defaults are 500 warmup steps and 800 retained draws per chain.
The table gives posterior medians and central 90% credible intervals
from the saved samples.


In [ ]:
import json
import numpy as np
from html import escape
from IPython.display import HTML, Image, Markdown, display
from example.inference.code.inference import OUTPUT, NAMES, recovery_rows, data_hash
with np.load(OUTPUT / "posterior.npz") as saved:
    draws = {name: saved[name] for name in NAMES}
diagnostics = json.loads((OUTPUT / "hmc.json").read_text())
assert diagnostics["data_sha256"] == data_hash(data)
display(Markdown(f"Maximum R̂: **{diagnostics['max_r_hat']:.4f}**; "
                 f"minimum ESS: **{diagnostics['min_effective_sample_size']:.0f}**; "
                 f"divergences: **{diagnostics['divergences']}**."))
rows = recovery_rows(data.truth, point, draws)
table = [
    '<table class="table"><thead><tr>',
    '<th scope="col">Parameter</th><th scope="col" style="text-align: right">Injected</th>',
    '<th scope="col" style="text-align: right">Median</th><th scope="col">90% interval</th>',
    '</tr></thead><tbody>',
]
for row in rows:
    table.append(
        f'<tr><th scope="row"><code>{escape(row["parameter"])}</code></th>'
        f'<td style="text-align: right">{row["injected"]:.6g}</td>'
        f'<td style="text-align: right">{row["median"]:.6g}</td>'
        f'<td>[{row["q05"]:.6g}, {row["q95"]:.6g}]</td></tr>'
    )
table.append('</tbody></table>')
display(HTML(''.join(table)))
display(Image(filename=str(OUTPUT / "recovery.png")))


Both panels use magnitudes. Residuals subtract **the true lens curve only**:
$\Delta m=-2.5\log_{10}(F_{\rm observed}/F_{\rm lens,true})$.
The simulated GP and white noise remain visible. The posterior median and
pointwise 90% band include the lens and conditional latent GP, with event,
flux, and GP uncertainty; they exclude new measurement noise.

Dark points average the residuals in 0.1-day bins, shorter than the injected
5-day rotation period and 20-day coherence time. Their error bars include white noise only. All 8640 original
observations enter the fit; binning is only for display.

## Corner

All twelve parameters are shown, including `a1`, both parallax components,
and all three GP hyperparameters. Orange lines mark truth; contours enclose 68% and 95% of
each two-dimensional posterior. Marginal dashed lines show 5/50/95% quantiles.


Download {download}`the full-size corner PDF <../../example/inference/outputs/corner.pdf>`.


In [ ]:
display(Image(filename=str(OUTPUT / "corner.png")))


## Trace

The two post-warmup chains are shown separately, with truth marked by dashed
lines. Convergence does not establish coverage: a single noise realization
can shift correlated event and flux estimates away from their injected values.
The selected positive-`u0` branch does not explore all parallax degeneracies.

In this short event, the north parallax component remains weakly constrained;
including parallax in the fit does not imply that both components are precisely measured.

The rotation period is constrained near the injected 5 days, while the
coherence time remains broad.


In [ ]:
display(Image(filename=str(OUTPUT / "trace.png")))


Outputs include the injected data, posterior samples, credible intervals,
diagnostics, and PNG/PDF figures. The [inference benchmark](../benchmarks.md#inference-efficiency)
compares VBM/emcee and singlens/NUTS on a separate nine-parameter event with
independent measurement noise.
